# DE-3 — Validação, Storage e Lineage

1. Checks no mock válido
2. Checks no mock quebrado
3. Parquet para DuckDB e consultas
4. Lineage: da proposta até o PDF oficial

## 1. Setup

In [1]:
import os, sys
from pathlib import Path

if not Path("src/quality").exists() and Path("../src/quality").exists():
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import duckdb
import pandas as pd
from src.quality import checks as c

print(Path.cwd())

/home/claude/mock


In [2]:
CHUNKS    = "data/mock/silver_mock.parquet"
MANIFEST  = "data/mock/documents_mock.csv"
PROPOSALS = "data/mock/proposals_mock.parquet"

# Na integração (16/09):
# CHUNKS   = "data/silver/chunks.parquet"
# MANIFEST = "data/bronze/metadata/documents.csv"

## 2. Carregar

In [3]:
chunks = pd.read_parquet(CHUNKS)
manifest = pd.read_csv(MANIFEST)

print(len(chunks), "chunks /", len(manifest), "documentos")
chunks.head()

10 chunks / 3 documentos


,chunk_id,document_id,page,section,text,n_chars,dataset_version
0,PRES_001_p003_c001,PRES_001,3,Apresentação,Este plano organiza nossas prioridades em cinc...,84,silver_v0
1,PRES_001_p012_c001,PRES_001,12,Saúde,Ampliar em 30% o número de equipes de Saúde da...,90,silver_v0
2,PRES_001_p012_c002,PRES_001,12,Saúde,Criar um programa nacional de regulação de fil...,97,silver_v0
3,PRES_001_p013_c001,PRES_001,13,Saúde,Garantir a distribuição de medicamentos de alt...,88,silver_v0
4,PRES_001_p018_c001,PRES_001,18,Educação,Expandir a oferta de creches em tempo integral...,80,silver_v0


## 3. Checks no mock válido

Os cinco checks do contrato. Todos devem passar.

In [4]:
for r in [c.chunk_id_nao_nulo_e_unico(chunks),
          c.document_id_existe_no_manifest(chunks, manifest),
          c.page_maior_ou_igual_a_um(chunks),
          c.text_nao_vazio(chunks),
          c.office_e_state_permitidos(manifest)]:
    print(r)

[PASSOU] chunk_id não nulo e único
[PASSOU] document_id existe no manifest
[PASSOU] page >= 1
[PASSOU] text não vazio
[PASSOU] office e state em valores permitidos


## 4. Checks no mock quebrado

Cinco violações plantadas de propósito: chunk_id duplicado, page = -1,
text vazio, document_id inexistente e office com maiúscula errada.
Cada uma deve ser acusada pelo seu check — e por nenhum outro.

In [5]:
chunks_bad = pd.read_parquet("data/mock/silver_broken.parquet")
manifest_bad = pd.read_csv("data/mock/documents_broken.csv")

for r in [c.chunk_id_nao_nulo_e_unico(chunks_bad),
          c.document_id_existe_no_manifest(chunks_bad, manifest),
          c.page_maior_ou_igual_a_um(chunks_bad),
          c.text_nao_vazio(chunks_bad),
          c.office_e_state_permitidos(manifest_bad)]:
    print(r)
    if r.detalhe:
        print("   ", r.detalhe)

[FALHOU] chunk_id não nulo e único — 2 violação(ões): PRES_001_p012_c001
    0 nulo(s), 2 linha(s) com ID repetido
[FALHOU] document_id existe no manifest — 1 violação(ões): PRES_999_p001_c001
    document_id não encontrado no manifest: PRES_999
[FALHOU] page >= 1 — 1 violação(ões): PRES_001_p000_c001
    valores encontrados: [-1]
[FALHOU] text não vazio — 1 violação(ões): PRES_002_p007_c003
    1 chunk(s) sem conteúdo aproveitável
[FALHOU] office e state em valores permitidos — 1 violação(ões): PRES_002
    office inválido: ['Presidente']


## 5. Storage — Parquet para DuckDB

In [6]:
con = duckdb.connect()

con.execute(f"CREATE TABLE documents AS SELECT * FROM read_csv_auto('{MANIFEST}')")
con.execute(f"CREATE TABLE chunks    AS SELECT * FROM read_parquet('{CHUNKS}')")
con.execute(f"CREATE TABLE proposals AS SELECT * FROM read_parquet('{PROPOSALS}')")

con.execute("SHOW TABLES").df()

,name
0,chunks
1,documents
2,proposals


### Chunks por documento

O PRES_003 aparece com zero chunks: o download falhou, então não há PDF
para o DE-2 processar.

In [7]:
con.execute("""
    SELECT d.document_id, d.candidate, COUNT(ch.chunk_id) AS n_chunks
    FROM documents d
    LEFT JOIN chunks ch ON ch.document_id = d.document_id
    GROUP BY d.document_id, d.candidate
    ORDER BY d.document_id
""").df()

,document_id,candidate,n_chunks
0,PRES_001,Ana Ribeiro Matos,5
1,PRES_002,Joaquim Torres Lemos,5
2,PRES_003,Marta Feijó Andrade,0


### Chunks por página

In [8]:
con.execute("""
    SELECT document_id, page, COUNT(*) AS n_chunks
    FROM chunks
    GROUP BY document_id, page
    ORDER BY document_id, page
""").df()

,document_id,page,n_chunks
0,PRES_001,3,1
1,PRES_001,12,2
2,PRES_001,13,1
3,PRES_001,18,1
4,PRES_002,5,1
5,PRES_002,7,2
6,PRES_002,9,1
7,PRES_002,22,1


## 6. Validar lendo do banco

Os checks recebem DataFrame, não caminho de arquivo. Por isso funcionam igual
sobre dados vindos do DuckDB — a validação não depende de onde o dado está.

In [9]:
chunks_db = con.execute("SELECT * FROM chunks").df()
manifest_db = con.execute("SELECT * FROM documents").df()

for r in [c.chunk_id_nao_nulo_e_unico(chunks_db),
          c.document_id_existe_no_manifest(chunks_db, manifest_db),
          c.page_maior_ou_igual_a_um(chunks_db),
          c.text_nao_vazio(chunks_db),
          c.office_e_state_permitidos(manifest_db)]:
    print(r)

[PASSOU] chunk_id não nulo e único
[PASSOU] document_id existe no manifest
[PASSOU] page >= 1
[PASSOU] text não vazio
[PASSOU] office e state em valores permitidos


## 7. Lineage

    proposal_id -> chunk_id -> document_id -> page -> source_url

In [10]:
con.execute("""
    SELECT p.proposal_id, p.chunk_id, ch.document_id, ch.page,
           d.candidate, d.source_url
    FROM proposals p
    JOIN chunks    ch ON ch.chunk_id   = p.chunk_id
    JOIN documents d  ON d.document_id = ch.document_id
    ORDER BY p.proposal_id
""").df()

,proposal_id,chunk_id,document_id,page,candidate,source_url
0,PRES_001_p012_c001_r01,PRES_001_p012_c001,PRES_001,12,Ana Ribeiro Matos,https://divulgacandcontas.tse.jus.br/planos/20...
1,PRES_001_p012_c002_r01,PRES_001_p012_c002,PRES_001,12,Ana Ribeiro Matos,https://divulgacandcontas.tse.jus.br/planos/20...
2,PRES_002_p007_c001_r01,PRES_002_p007_c001,PRES_002,7,Joaquim Torres Lemos,https://divulgacandcontas.tse.jus.br/planos/20...


### Uma proposta específica

In [11]:
linha = con.execute("""
    SELECT p.text AS proposta, ch.document_id, ch.page,
           d.candidate, d.party, d.source_url
    FROM proposals p
    JOIN chunks    ch ON ch.chunk_id   = p.chunk_id
    JOIN documents d  ON d.document_id = ch.document_id
    WHERE p.proposal_id = 'PRES_001_p012_c001_r01'
""").df().iloc[0]

print("proposta :", linha["proposta"])
print("documento:", linha["document_id"], "-", linha["candidate"], f"({linha['party']})")
print("página   :", linha["page"])
print("fonte    :", linha["source_url"])

proposta : Ampliar em 30% o número de equipes de Saúde da Família.
documento: PRES_001 - Ana Ribeiro Matos (PDA)
página   : 12
fonte    : https://divulgacandcontas.tse.jus.br/planos/2026/PRES/ana-ribeiro-matos.pdf


## 8. O teste da integração (16/09)

Trocar CHUNKS e MANIFEST pelos caminhos reais e rodar tudo de novo. Se os mesmos
checks continuarem funcionando sem alterar código, o contrato estava certo.

In [12]:
con.close()